# H2 · Growth and catch-up: do firms in poorer countries grow faster?

> **Hypothesis H2:** Companies in lower-income countries grow their revenue faster than companies in richer countries, and this is more than just higher inflation.

| | |
|---|---|
| **Why we ask** | In the first scan, **median revenue growth** was linked to 12 of 35 country features, mostly governance and wealth, with higher growth where institutions are weaker. That is the classic *catch-up* pattern, but revenue is measured in local currency, so inflation could be doing the work. |
| **Prediction** | **Negative** association between the development index and *real* revenue growth (growth net of national inflation). |
| **Data** | Yahoo company-years with two consecutive fiscal years (about 5,000). *Development index* = average of GDP per person, life expectancy, internet use, rule of law, government effectiveness and corruption control (previous year, SDs across countries). *Real growth* = (1 + revenue growth) / (1 + national inflation in the same year) − 1. |
| **Primary test (fixed before looking at the result)** | `real revenue growth (pp) ~ development index + sector + size rank + fiscal year`, errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Growth by country
Median revenue growth of each country's companies, before and after removing national inflation, against the development index. If catch-up is real, the real-growth line should still slope downwards.

In [2]:
g = fy.groupby("iso3")
gr = pd.DataFrame({"nominal growth (pp)": g.growth_pp.median(), "real growth (pp)": g.growth_real_pp.median(), "national inflation (%)": g.inflation_now.median(),
                   "national GDP growth (%)": g.gdp_growth_now.median(), "firms with growth": g.apply(lambda d: d.growth_pp.notna().groupby(d.company_id).any().sum())})
gr = gr[gr["firms with growth"] >= MIN_FIRMS].join(country["dev_idx"].rename("development index")).join(dim[["name", "region"]])
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{c} (ρ = {spearman(gr['development index'], gr[c])[0]:+.2f}, n = {gr[c].notna().sum()})" for c in ["nominal growth (pp)", "real growth (pp)"]])
for k, c in enumerate(["nominal growth (pp)", "real growth (pp)"], 1):
    d = gr[["development index", c]].dropna()
    fig.add_trace(go.Scatter(x=d["development index"], y=d[c], mode="markers+text", text=d.index, textposition="top center", textfont_size=8,
                             marker=dict(size=7, color="#4C78A8"), showlegend=False), row=1, col=k)
    b, a = np.polyfit(d["development index"], d[c], 1); xs = np.array([d["development index"].min(), d["development index"].max()])
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#E45756"), showlegend=False), row=1, col=k)
fig.update_xaxes(title_text="development index (SD)"); fig.update_layout(height=470, title="Median company revenue growth vs development, one point per country")
fig.show()
gr.sort_values("development index").round(2)[["name", "nominal growth (pp)", "real growth (pp)", "national inflation (%)", "national GDP growth (%)", "development index"]]

,name,nominal growth (pp),real growth (pp),national inflation (%),national GDP growth (%),development index
iso3,,,,,,
IND,India,9.280,3.400,5.650,7.210,-2.190
IDN,Indonesia,6.570,4.530,3.670,5.050,-1.940
MEX,Mexico,7.750,2.760,5.530,3.110,-1.820
ZAF,South Africa,6.040,0.680,6.080,0.810,-1.810
BRA,Brazil,9.000,4.570,4.590,3.240,-1.480
CHN,China,5.190,5.670,0.230,4.960,-1.160
ROU,Romania,3.480,-4.590,10.400,2.260,-0.910
HUN,Hungary,6.720,-0.650,14.610,0.710,-0.620
GRC,Greece,10.690,7.230,3.460,2.140,-0.530


## 2 · Primary test (firm level)
Compares companies in the same sector, of similar relative size and in the same fiscal year. The coefficient is the change in **real revenue growth, in percentage points per year**, for a country that is +1 SD more developed.

In [3]:
primary = wild_cluster("dev_z", fy, outcome="growth_real_pp", B=1999)
med = fy.growth_real_pp.median()
print(f"Typical real revenue growth: {med:.1f}% a year (median).")
print(f"+1 SD development → {primary['coef']:+.2f} pp real growth (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}).")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f} | {primary['clusters']} countries, {primary['n']:,} company-years")

Typical real revenue growth: 1.8% a year (median).
+1 SD development → -1.41 pp real growth (95% CI -3.07 to +0.26).
p (cluster-robust) = 0.1068 | p (wild bootstrap) = 0.1450 | 34 countries, 5,039 company-years


### Is it inflation, or national growth?
Same model with other outcomes and controls. If the effect shrinks a lot when national GDP growth is controlled, it is mainly that poorer countries are growing faster overall, not something specific to their companies.

In [4]:
specs = [("nominal revenue growth (pp)", "growth_pp", ""), ("real revenue growth (pp)  [primary]", "growth_real_pp", ""),
         ("real growth, + national GDP growth", "growth_real_pp", "gdp_growth_now"), ("nominal growth, + national inflation", "growth_pp", "inflation_now")]
rows = []
for lab, out, extra in specs:
    r = wild_cluster("dev_z", fy, outcome=out, extra=extra, B=999)
    rows.append({"specification": lab, "development coef (pp)": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "firm-years": r["n"]})
secondary = pd.DataFrame(rows).set_index("specification")
secondary

,development coef (pp),ci_low,ci_high,p (wild),firm-years
specification,,,,,
nominal revenue growth (pp),-1.751,-2.948,-0.554,0.029,7493
real revenue growth (pp) [primary],-1.409,-3.075,0.257,0.138,5039
"real growth, + national GDP growth",-0.313,-1.503,0.876,0.639,5039
"nominal growth, + national inflation",-1.822,-3.198,-0.446,0.052,5039


## 3 · Does firm growth follow national growth?
Firm growth should move with the economy, but how much of that is *between countries* (fast economies have fast firms) and how much *within a country over time*? Country-year medians are compared in levels and after removing each country's own average.

In [5]:
cyr = fy.groupby(["iso3", "fiscal_year"]).agg(g=("growth_real_pp", "median"), gdp=("gdp_growth_now", "first"), n=("growth_real_pp", "size")).query("n >= 8").dropna()
cyr["g_within"] = cyr.g - cyr.groupby("iso3").g.transform("mean"); cyr["gdp_within"] = cyr.gdp - cyr.groupby("iso3").gdp.transform("mean")
rb, pb, nb = spearman(cyr.groupby("iso3").g.mean(), cyr.groupby("iso3").gdp.mean())
rw, pw, nw = spearman(cyr.g_within, cyr.gdp_within)
rt, pt, nt = spearman(cyr.g, cyr.gdp)
firm = fit_cluster("growth_real_pp ~ gdp_growth_now + " + CONTROLS, fy.dropna(subset=["growth_real_pp", "gdp_growth_now", "size_rank"]))
print(f"Country-year medians (n={nt}): ρ = {rt:+.2f}. Between countries (country means, n={nb}): ρ = {rb:+.2f} (p={pb:.3f}). Within a country over time (n={nw}): ρ = {rw:+.2f} (p={pw:.3f}).")
print(f"Firm level: +1 pp national GDP growth → {firm.params['gdp_growth_now']:+.2f} pp real revenue growth (cluster-robust p = {firm.pvalues['gdp_growth_now']:.3f}).")
px.scatter(cyr.reset_index().assign(country=lambda d: d.iso3.map(names)), x="gdp", y="g", color="country", title="Median real company growth vs national GDP growth, by country and year", labels={"gdp": "national GDP growth (%)", "g": "median real firm growth (pp)"}, height=430).update_layout(showlegend=False)

Country-year medians (n=92): ρ = +0.34. Between countries (country means, n=34): ρ = +0.32 (p=0.062). Within a country over time (n=92): ρ = -0.02 (p=0.833).
Firm level: +1 pp national GDP growth → +1.20 pp real revenue growth (cluster-robust p = 0.001).


## 4 · Does profit keep up with growth?
If poorer-country firms grow faster but earn less, growth is not translating into returns.

In [6]:
cg = gr.join(fy.groupby("iso3").roa_pp.median().rename("median ROA (pp)"))
r1, p1, n1 = spearman(cg["real growth (pp)"], cg["median ROA (pp)"])
fy["growth_q"] = fy.groupby("sector").growth_real_pp.transform(lambda s: pd.qcut(s, 5, labels=False, duplicates="drop") + 1)
byq = fy.dropna(subset=["growth_q"]).groupby("growth_q").roa_pp.agg(["median", "count"])
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Countries: real growth vs ROA (ρ = {r1:+.2f}, n = {n1})", "Companies: ROA by growth quintile (within sector)"))
fig.add_trace(go.Scatter(x=cg["real growth (pp)"], y=cg["median ROA (pp)"], mode="markers+text", text=cg.index, textposition="top center", textfont_size=8, marker=dict(size=7), showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=byq.index.astype(int), y=byq["median"], marker_color="#4C78A8", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="median real growth (pp)", row=1, col=1); fig.update_xaxes(title_text="growth quintile (1 = slowest)", row=1, col=2)
fig.update_yaxes(title_text="median ROA (pp)"); fig.update_layout(height=440); fig.show()
byq.round(2)

,median,count
growth_q,,
1.000,2.310,1016
2.000,3.850,1005
3.000,4.100,1005
4.000,4.250,1004
5.000,4.130,1009


## 5 · Robustness
The primary test in seven samples (wild bootstrap, 499 draws each).

In [7]:
rob, rob_share = robustness("dev_z", "growth_real_pp", -1)
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="Development → real revenue growth (pp per +1 SD) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Expected sign (negative) in {rob_share:.0%} of samples.")
rob

Expected sign (negative) in 100% of samples.


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,-1.409,-3.075,0.257,0.152,34,5039,True
excluding financials & real estate,-1.101,-3.183,0.981,0.368,34,3983,True
excluding small-sample countries,-1.371,-3.051,0.309,0.164,29,4826,True
"excluding USA, China, Japan",-1.064,-2.687,0.560,0.274,31,4440,True
fiscal years 2023 and later,-1.447,-3.138,0.243,0.156,34,4954,True
EU countries only,-1.184,-5.664,3.296,0.674,21,2512,True
non-EU countries only,-1.026,-2.882,0.831,0.336,13,2527,True


## 6 · Replication on ESEF
EU companies only (20 countries, so less variation in development), no sector control.

In [8]:
fe = prep.load_firms(con, country, source="ESEF")
rep = wild_cluster("dev_z", fe, outcome="growth_real_pp", controls="size_rank + C(fiscal_year)", B=1999)
ge = fe.groupby("iso3").agg(g=("growth_real_pp", "median"), firms=("company_id", "nunique")).query("firms >= 10").join(country["dev_idx"].rename("dev"))
rho_e, p_e, n_e = spearman(ge.dev, ge.g)
print(f"ESEF, firm level ({rep['n']:,} company-years, {rep['clusters']} countries): +1 SD development → {rep['coef']:+.2f} pp real growth (95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")
print(f"ESEF, country level: ρ between median real growth and development = {rho_e:+.2f} (p = {p_e:.3f}, n = {n_e})")

ESEF, firm level (2,424 company-years, 19 countries): +1 SD development → -2.90 pp real growth (95% CI -6.80 to +1.00), wild p = 0.220
ESEF, country level: ρ between median real growth and development = -0.39 (p = 0.098, n = 19)


## 7 · Verdict

In [9]:
v = utils.verdict(primary["coef"], primary["p_wild"], -1, rob_share, primary["clusters"])
rep_same = bool(np.sign(rep["coef"]) == -1)
control_gdp = secondary.loc["real growth, + national GDP growth"]
lines = [
 f"Primary: +1 SD development → {primary['coef']:+.2f} pp real revenue growth (typical {med:.1f}%), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} company-years.",
 f"Robustness: expected sign in {rob_share:.0%} of seven samples. ESEF replication: {rep['coef']:+.2f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"Controlling for national GDP growth the coefficient is {control_gdp['development coef (pp)']:+.2f} (p = {control_gdp['p (wild)']:.3f}).",
 f"Profit vs growth at country level: ρ = {r1:+.2f} (n = {n1}).",
 f"VERDICT H2: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H2", id="H2", title="Growth and catch-up", statement="Firms in lower-income countries grow real revenue faster.",
                  expected_sign=-1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp real revenue growth per +1 SD development",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.2_growth_and_catch_up.ipynb", notes=lines)

Primary: +1 SD development → -1.41 pp real revenue growth (typical 1.8%), wild p = 0.145, 34 countries, 5,039 company-years.
Robustness: expected sign in 100% of seven samples. ESEF replication: -2.90 pp (wild p = 0.220), same direction.
Controlling for national GDP growth the coefficient is -0.31 (p = 0.639).
Profit vs growth at country level: ρ = +0.14 (n = 34).
VERDICT H2: NOT SUPPORTED


### Limits
- Revenue growth is in **local currency**; the inflation adjustment uses national consumer prices, which is only an approximation of each firm's price changes. Exchange-rate moves and acquisitions also change reported revenue.
- Only companies with two consecutive years of revenue are included (about half of the sample), and very few years overlap with the 2020–2021 shock.
- The development index and institutions are strongly correlated across countries, so the result cannot separate wealth from institutions.
- Association, not causation.

In [10]:
con.close()